# NES-VMC: 扩展系统哈密顿量构建（完整实现）

本 notebook 实现了 NES-VMC 算法中的核心部分：构建扩展系统的哈密顿量，并完成完整的训练流程。

## 理论背景

NES-VMC (Natural Excited State Variational Monte Carlo) 算法的核心思想是将原系统前 K 个激发态的求解问题，等价地转化为一个"扩展系统"的基态求解问题。

扩展系统的哈密顿量是原哈密顿量的直和形式：

$$H_{\text{extended}} = H \otimes I \otimes I \otimes \cdots \otimes I + I \otimes H \otimes I \otimes \cdots \otimes I + \cdots + I \otimes I \otimes \cdots \otimes H$$

这是一个 K 个项的和，每一项都是原哈密顿量作用在其中一个子系统上。

## 1. 导入必要的库

In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from functools import partial
from typing import Tuple, Any

# 设置 JAX 浮点精度
jax.config.update("jax_enable_x64", True)

print(f"NetKet version: {nk.__version__}")
print(f"JAX version: {jax.__version__}")

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NetKet version: 3.18
JAX version: 0.5.3


## 2. 设置分子系统（H₂ 分子）

In [2]:
# H₂ 分子定义
bond_length = 1.4  # Bohr
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print("=" * 60)
print("H₂ FCI 基准能量 (STO-3G)")
print("=" * 60)
for i, e in enumerate(E_fcis):
    exc_eV = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc_eV:.4f} eV")

H₂ FCI 基准能量 (STO-3G)
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV


## 3. 定义原始哈密顿量和 Hilbert 空间

In [3]:
# 将 PySCF 分子转为 NetKet 离散算符
ha_original = nkx.operator.from_pyscf_molecule(mol)

# 转换为 JAX 兼容的算子（重要！）
try:
    ha_original = ha_original.to_jax_operator()
    print("✓ 成功转换为 JAX 兼容算子")
except Exception as e:
    print(f"✗ 无法转换为 JAX 算子: {e}")
    print("继续使用原始算子")

# 原始 Hilbert 空间
hi_original = nkx.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1),
)

print(f"\n原始 Hilbert 空间: {hi_original}")
print(f"原始 Hilbert 空间大小: {hi_original.size}")
print(f"原始 Hilbert 空间状态数: {hi_original.n_states}")
print(f"\n原始哈密顿量类型: {type(ha_original)}")

✓ 成功转换为 JAX 兼容算子

原始 Hilbert 空间: SpinOrbitalFermions(n_orbitals=2, s=1/2, n_fermions=2, n_fermions_per_spin=(1, 1))
原始 Hilbert 空间大小: 4
原始 Hilbert 空间状态数: 4

原始哈密顿量类型: <class 'netket.experimental.operator._particle_number_conserving_fermionic._operators.ParticleNumberAndSpinConservingFermioperator2nd'>


/var/folders/8x/k_m4pmb11437ktb_r6tjzt2c0000gn/T/ipykernel_35747/359029433.py:13: DeprecationWarning: netket.experimental.hilbert.SpinOrbitalFermions is deprecated: use netket.hilbert.SpinOrbitalFermions (netket >= 3.12)
  hi_original = nkx.hilbert.SpinOrbitalFermions(


## 4. 定义扩展 Hilbert 空间

In [4]:
# NES-VMC 参数
K = 2  # 需要计算的态数（基态 + 1 个激发态）

# 扩展 Hilbert 空间：K 个副本的直积
hi_extended = hi_original ** K

print(f"扩展 Hilbert 空间: {hi_extended}")
print(f"扩展 Hilbert 空间大小: {hi_extended.size}")
print(f"扩展 Hilbert 空间状态数: {hi_extended.n_states}")
print(f"\n扩展 Hilbert 空间状态示例:")
states = hi_extended.all_states()
print(f"总状态数: {states.shape[0]}")
print(f"状态形状: {states.shape}")
print(f"\n前几个状态:")
for i in range(min(5, states.shape[0])):
    print(f"  状态 {i}: {states[i]}")

扩展 Hilbert 空间: SpinOrbitalFermions(n_orbitals=2, s=1/2, n_fermions=2, n_fermions_per_spin=(1, 1))⊗SpinOrbitalFermions(n_orbitals=2, s=1/2, n_fermions=2, n_fermions_per_spin=(1, 1))
扩展 Hilbert 空间大小: 8
扩展 Hilbert 空间状态数: 16

扩展 Hilbert 空间状态示例:
总状态数: 16
状态形状: (16, 8)

前几个状态:
  状态 0: [0 1 0 1 0 1 0 1]
  状态 1: [0 1 0 1 0 1 1 0]
  状态 2: [0 1 0 1 1 0 0 1]
  状态 3: [0 1 0 1 1 0 1 0]
  状态 4: [0 1 1 0 0 1 0 1]


## 5. 构建扩展系统哈密顿量（矩阵表示）

In [5]:
def build_extended_hamiltonian_matrix(hi_original, hi_extended, original_hamiltonian, K):
    """
    使用矩阵表示构建扩展系统哈密顿量
    
    参数:
        hi_original: 原始 Hilbert 空间
        hi_extended: 扩展 Hilbert 空间
        original_hamiltonian: 原始哈密顿量
        K: 子系统数量
    
    返回:
        H_extended: 扩展系统的哈密顿量矩阵
    """
    # 获取原始 Hilbert 空间的所有状态
    states_original = hi_original.all_states()
    n_states_original = states_original.shape[0]
    
    print(f"构建扩展哈密顿量矩阵...")
    print(f"原始系统状态数: {n_states_original}")
    print(f"扩展系统状态数: {hi_extended.n_states}")
    
    # 构建原始哈密顿量的矩阵表示
    H_original = jnp.zeros((n_states_original, n_states_original), dtype=complex)
    
    for i, state in enumerate(states_original):
        # 获取连接组态和矩阵元
        conn_states, mels = original_hamiltonian.get_conn(state)
        
        # 填充矩阵
        for conn_state, mel in zip(conn_states, mels):
            # 找到连接组态的索引
            j = hi_original.states_to_numbers(conn_state)
            H_original = H_original.at[i, j].set(mel)
    
    print(f"原始哈密顿量矩阵形状: {H_original.shape}")
    
    # 构建扩展系统的哈密顿量
    # H_extended = H ⊗ I ⊗ ... ⊗ I + I ⊗ H ⊗ ... ⊗ I + ... 
    
    # 单位矩阵
    I = jnp.eye(n_states_original, dtype=complex)
    
    # 初始化扩展哈密顿量
    H_extended = jnp.zeros((hi_extended.n_states, hi_extended.n_states), dtype=complex)
    
    # 对每个子系统构建项
    for i in range(K):
        # 构建第 i 项：I ⊗ I ⊗ ... ⊗ H ⊗ ... ⊗ I
        # 其中 H 在第 i 个位置
        
        term = jnp.array([[1.0]], dtype=complex)
        
        for j in range(K):
            if j == i:
                term = jnp.kron(term, H_original)
            else:
                term = jnp.kron(term, I)
        
        H_extended = H_extended + term
    
    print(f"扩展哈密顿量矩阵形状: {H_extended.shape}")
    
    return H_extended

In [6]:
# 构建扩展哈密顿量
H_extended_matrix = build_extended_hamiltonian_matrix(
    hi_original, hi_extended, ha_original, K
)

# 验证扩展哈密顿量的性质
print("\n" + "=" * 60)
print("扩展哈密顿量验证")
print("=" * 60)

# 检查厄米性
is_hermitian = jnp.allclose(H_extended_matrix, H_extended_matrix.conj().T)
print(f"是否厄米: {is_hermitian}")

# 计算本征值
eigenvalues = jnp.linalg.eigvalsh(H_extended_matrix)
print(f"\n扩展哈密顿量的本征值:")
for i, ev in enumerate(eigenvalues[:8]):
    print(f"  λ_{i} = {ev:.8f}")

构建扩展哈密顿量矩阵...
原始系统状态数: 4
扩展系统状态数: 16
原始哈密顿量矩阵形状: (4, 4)
扩展哈密顿量矩阵形状: (16, 16)

扩展哈密顿量验证
是否厄米: True

扩展哈密顿量的本征值:
  λ_0 = -2.03093650
  λ_1 = -1.89089619
  λ_2 = -1.89089619
  λ_3 = -1.75085588
  λ_4 = -1.44485201
  λ_5 = -1.44485201
  λ_6 = -1.30481170
  λ_7 = -1.30481170


## 6. 构建采样器

In [7]:
# 使用简单的 LocalRule
single_rule = nk.sampler.rules.LocalRule()

tensor_rule = nk.sampler.rules.TensorRule(
    hilbert=hi_extended,
    rules=[single_rule] * K
)

sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_extended,
    rule=tensor_rule,
    n_chains=16,
    sweep_size=16,
    reset_chains=True,
)

print(f"采样器: {sampler}")

采样器: MetropolisSampler(rule = TensorRule(hilbert=SpinOrbitalFermions(n_orbitals=2, s=1/2, n_fermions=2, n_fermions_per_spin=(1, 1))⊗SpinOrbitalFermions(n_orbitals=2, s=1/2, n_fermions=2, n_fermions_per_spin=(1, 1)), rules=(LocalRule(), LocalRule())), n_chains = 16, sweep_size = 16, reset_chains = True, machine_power = 2, dtype = int8)


## 7. 神经网络模型定义

In [8]:
class SingleStateAnsatz(nnx.Module):
    """单态 Ansatz"""
    def __init__(self, n_spin_orbitals: int, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)


class NESTotalAnsatz(nnx.Module):
    """NES 总 Ansatz"""
    def __init__(self, n_spin_orbitals: int, n_states: int = K, hidden_dim: int = 16):
        super().__init__()
        self.n_states = n_states
        self.n_spin_orbitals = n_spin_orbitals

        key = jax.random.key(42)
        keys = jax.random.split(key, n_states)

        self.single_ansatz_list = [
            SingleStateAnsatz(
                n_spin_orbitals,
                hidden_dim,
                rngs=nnx.Rngs(keys[i])
            )
            for i in range(n_states)
        ]

    def __call__(self, x_batch):
        """x_batch: (n_states, n_spin_orbitals) -> (psi_total, M)"""
        K_state = self.n_states
        M = []
        for i in range(K_state):
            row = [self.single_ansatz_list[j](x_batch[i]) for j in range(K_state)]
            M.append(jnp.array(row))
        M = jnp.stack(M)
        psi_total = jnp.linalg.det(M)
        return psi_total, M


# 实例化总 Ansatz
n_spin_orbitals = hi_original.size
total_ansatz = NESTotalAnsatz(
    n_spin_orbitals=n_spin_orbitals,
    n_states=K,
    hidden_dim=16,
)

graph_def, params = nnx.split(total_ansatz)
print(f"模型参数数量: {sum(p.size for p in jax.tree.leaves(params))}")

模型参数数量: 738


## 8. 局部能量计算

In [9]:
def Ham_psi(ha, single_ansatz, x):
    """单态 Ansatz 在组态 x 上的 Hψ 值"""
    x = x.squeeze()
    x_primes, mels = ha.get_conn_padded(x)
    psi_vals = jax.vmap(single_ansatz)(x_primes)
    return jnp.sum(mels * psi_vals)


def Ham_Psi(ha, total_ansatz, x_batch):
    """计算 HΨ 矩阵，形状 (K, K)"""
    K = total_ansatz.n_states
    H_mat = []
    for i in range(K):
        row = []
        for j in range(K):
            v = Ham_psi(ha, total_ansatz.single_ansatz_list[j], x_batch[i])
            row.append(v)
        H_mat.append(row)
    return jnp.array(H_mat, dtype=complex)


def compute_local_energy(ha, total_ansatz, x_batch, eps=1e-6):
    """
    计算局部能量矩阵及其迹
    x_batch: (K, n_spin_orbitals)
    返回 (trace_real, el_mat)
    """
    psi, M = total_ansatz(x_batch)
    det_val = jnp.linalg.det(M)
    cond = jnp.abs(det_val) < 1e-4
    actual_eps = jnp.where(cond, 1e-4, eps)
    M_reg = M + actual_eps * jnp.eye(M.shape[0], dtype=M.dtype)
    Hp = Ham_Psi(ha, total_ansatz, x_batch)
    el_mat = jnp.linalg.solve(M_reg, Hp)
    return jnp.trace(el_mat).real, el_mat


compute_local_energy_batch = jax.vmap(
    compute_local_energy,
    in_axes=(None, None, 0, None),
    out_axes=(0, 0)
)


def compute_average_local_energy(ha, model, samples, eps=1e-6):
    """
    samples: (n_samples, K, n_spin_orbitals)
    """
    tr_els, el_mats = compute_local_energy_batch(ha, model, samples, eps)
    tr_avg = tr_els.mean()
    el_mat_avg = el_mats.mean(axis=0)
    return tr_avg, el_mat_avg

## 9. 损失函数与梯度

In [10]:
def loss_fn(params, ha, x_batch):
    graphdef, variables = params
    model = nnx.merge(graphdef, variables)
    tr_avg, _ = compute_average_local_energy(ha, model, x_batch, eps=1e-6)
    return tr_avg


value_and_grad = jax.value_and_grad(loss_fn, argnums=0)

## 10. Forward 函数（供采样器使用）

In [11]:
def forward(params, x_batch):
    """
    x_batch: (n_chains, K * n_spin_orbitals)
    返回: (n_chains,) 每个元素的 log|Ψ|
    """
    graphdef, variables = params
    model = nnx.merge(graphdef, variables)
    n_chains = x_batch.shape[0]
    K_state = model.n_states
    n_spin = model.n_spin_orbitals
    x_reshaped = x_batch.reshape(n_chains, K_state, n_spin)
    
    def single_logpsi(x):
        psi, _ = model(x)
        return jnp.log(jnp.abs(psi) + 1e-12)
    
    log_psi_batch = jax.vmap(single_logpsi)(x_reshaped)
    return log_psi_batch

## 11. 初始化模型、采样器、优化器

In [12]:
sampler_state = sampler.init_state(forward, (graph_def, params), seed=1)

optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(learning_rate=1e-2)
)
opt_state = optimizer.init(params)

print("✓ 初始化完成")

✓ 初始化完成


## 12. 训练循环

In [13]:
n_iter = 100
chain_length = 200
loss_record = []

print("\n" + "=" * 60)
print("开始训练 NES-VMC（使用扩展哈密顿量）")
print("=" * 60)

for step in range(n_iter):
    sampler_state = sampler.reset(forward, (graph_def, params), sampler_state)
    samples, sampler_state = sampler.sample(
        forward, (graph_def, params), state=sampler_state, chain_length=chain_length
    )
    samples_flat = samples.reshape(-1, K, n_spin_orbitals)
    
    loss_val, grads = value_and_grad((graph_def, params), ha_original, samples_flat)
    loss_record.append(loss_val)
    
    grad_graph, grad_vars = grads
    updates, opt_state = optimizer.update(grad_vars, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    if step % 20 == 0:
        print(f"\nStep {step:4d} | Trace of local energy matrix: {loss_val:.6f} Ha")
        
        _, E_mat_avg = compute_average_local_energy(ha_original, nnx.merge(graph_def, params), samples_flat)
        eigvals = jnp.linalg.eigvals(E_mat_avg).real
        eigvals_sorted = jnp.sort(eigvals)
        print(f"Epoch {step:4d} | Loss = {loss_val:.6f} | Energies: {eigvals_sorted}")

total_ansatz = nnx.merge(graph_def, params)
print("\n✓ 训练完成")


开始训练 NES-VMC（使用扩展哈密顿量）

Step    0 | Trace of local energy matrix: -1.638198 Ha
Epoch    0 | Loss = -1.638198 | Energies: [-0.91025878 -0.74153806]

Step   20 | Trace of local energy matrix: -1.626789 Ha
Epoch   20 | Loss = -1.626789 | Energies: [-0.83349685 -0.7896584 ]

Step   40 | Trace of local energy matrix: -1.631974 Ha
Epoch   40 | Loss = -1.631974 | Energies: [-0.89910511 -0.73778704]

Step   60 | Trace of local energy matrix: -1.714394 Ha
Epoch   60 | Loss = -1.714394 | Energies: [-0.94622962 -0.77806089]

Step   80 | Trace of local energy matrix: -1.580962 Ha
Epoch   80 | Loss = -1.580962 | Energies: [-0.89726422 -0.68426383]

✓ 训练完成


## 13. 最终采样与能量计算

In [14]:
print("\n" + "=" * 60)
print("最终采样，计算能量矩阵...")
print("=" * 60)

final_samples, _ = sampler.sample(
    forward, (graph_def, params), state=sampler_state, chain_length=1000
)
final_samples_flat = final_samples.reshape(-1, K, n_spin_orbitals)

_, el_mat_avg = compute_average_local_energy(ha_original, total_ansatz, final_samples_flat, eps=1e-6)
el_mat_sym = (el_mat_avg + el_mat_avg.conj().T) / 2
eigen_energies = jnp.linalg.eigvalsh(el_mat_sym).real
eigen_energies = jnp.sort(eigen_energies)

print("\n" + "=" * 60)
print("NES-VMC 计算得到的激发态能量 (Ha)")
print("=" * 60)
for i, e in enumerate(eigen_energies):
    exc = (e - eigen_energies[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

print("\nFCI 基准:")
for i, e in enumerate(E_fcis[:K]):
    exc_eV = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc_eV:.4f} eV")


最终采样，计算能量矩阵...

NES-VMC 计算得到的激发态能量 (Ha)
E0 = -0.91292832 Ha  |  激发能: 0.0000 eV
E1 = -0.67633217 Ha  |  激发能: 6.4381 eV

FCI 基准:
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV


## 14. 总结

本 notebook 实现了完整的 NES-VMC 算法，包括：

1. **扩展系统哈密顿量的构建**：使用矩阵表示构建了直和形式的哈密顿量
2. **JAX 兼容性处理**：将 Numba-based 算子转换为 JAX 兼容算子
3. **神经网络 Ansatz**：实现了基于行列式的多态 Ansatz
4. **训练流程**：完成了完整的 VMC 训练循环

### 改进方向

1. 增加训练次数和采样数量
2. 调整学习率和优化器参数
3. 尝试更复杂的神经网络架构
4. 使用更合适的采样规则（如 FermionHopRule）